# H9 — Three missions sharing one antenna
An illustrative crewed lander and two CLPS-class robotic missions share one assumed DSS-43 channel. This notebook uses archived terrain-relative Sun/Earth analyses and R2 DSN geometry for 4–5 November 2026. Mission priorities, contact demand and station availability are declared assumptions. It is not a NASA schedule, antenna reservation or flight/energy certificate.

**Run All is offline: it does not connect to OpenQuantum or spend credits.** Recorded hardware is loaded only when an H9 fixture exists. Use the Lunar H7 kernel.

In [1]:
import json, sys, os
from pathlib import Path
from html import escape
from IPython.display import display, HTML
import numpy as np
ROOT = Path(os.environ.get("LUNAR_REPO_ROOT", Path.cwd())).resolve()
while not (ROOT / "shared/hybrid.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "shared/hybrid.py").exists(), "Start from the repository or set LUNAR_REPO_ROOT"
sys.path[:0] = [str(ROOT), str(ROOT / "backend")]
from app.shared_antenna import (CampaignRequest, load_campaign_evidence, build_campaign,
    campaign_catalog, compare_campaign, campaign_timeline, campaign_hardware)
from app.hybrid_planning import select_independent, solve_exact, validate_plan
from app.hybrid_quantum import run_qaoa
from shared.hybrid import fingerprint, Plan

def table(rows):
    if not rows:
        print("No rows")
        return
    keys = list(rows[0])
    display(HTML("<table><thead><tr>" + "".join("<th>"+escape(k)+"</th>" for k in keys) +
        "</tr></thead><tbody>" + "".join("<tr>"+"".join("<td>"+escape(str(row[k]))+"</td>" for k in keys)+"</tr>" for row in rows)+"</tbody></table>"))
print("Repository:", ROOT)
print("New provider jobs: 0; new credits spent: 0")

Repository: /Users/tk/.codex/worktrees/task1-ingestion-cache/Nasa-Space-Apps-Challenge
New provider jobs: 0; new credits spent: 0


## 1. Examine the evidence before optimizing
Each candidate is bound to its saved analysis ID, MOON_ME site, UTC interval and R2 result. Checksums cover the archived source projection. The second site's label remains `custom`: the coordinates are explicit and are not promoted to an officially named landing site.

All selected contact intervals are dark. Sunlight is an observed metric here, not a hard solar-power requirement. A practical powered mission requires a separately modeled energy budget.

In [2]:
evidence = load_campaign_evidence()
catalog = campaign_catalog()
table([dict(option=o["index"], site=o["site"]["site_id"], latitude_deg=o["site"]["latitude_deg"],
    longitude_deg_east=o["site"]["longitude_deg_east"], start_utc=o["window"]["start_utc"],
    end_utc=o["window"]["end_utc"], geometry=o["contact_state"], sunlight_s=o["sunlight_s"],
    analysis_id=o["source"]["analysis_id"]) for o in catalog["options"]])
assert catalog["options"][2]["contact_state"] == "unavailable"
assert all(o["sunlight_s"] == 0 for o in catalog["options"])

option,site,latitude_deg,longitude_deg_east,start_utc,end_utc,geometry,sunlight_s,analysis_id
1,mons-mouton-feature-center,-84.6,31.0,2026-11-04T00:00:00Z,2026-11-04T00:30:00Z,available,0.0,e430cbda-c6fb-5828-b105-f16eb6579441
2,custom,-84.40115,21.95459,2026-11-04T01:00:00Z,2026-11-04T01:30:00Z,available,0.0,c817a988-79c8-4cfe-9a19-71b5a9424838
3,mons-mouton-feature-center,-84.6,31.0,2026-11-05T00:00:00Z,2026-11-05T00:30:00Z,unavailable,0.0,9481b76b-7332-4403-9844-d04779fb02c3
4,custom,-84.40115,21.95459,2026-11-05T01:00:00Z,2026-11-05T01:30:00Z,available,0.0,429b549c-07b4-48f3-b087-c918edc1c315


## 2. Build the exact shared input
Three mandatory missions, four alternatives each: 64 assignments. The preference metric uses mission multipliers 3/2/1 and alternative values 4/3/2/1. These weights express a teaching preference, not real spacecraft specifications. DSS-43 capacity is explicitly assumed to be one concurrent channel; geometry is a separate per-candidate gate.

In [3]:
settings = CampaignRequest()
problem = build_campaign(settings, evidence)
digest = fingerprint(problem)
print("Input fingerprint:", digest)
print("Missions:", len(problem.missions), "Candidates:", len(problem.candidates))
table([dict(mission=c.mission_id, option=c.option_id, preference=c.metrics["preference"],
    sunlight_s=c.metrics["sunlight_s"], start_utc=c.window.start_utc.isoformat()) for c in problem.candidates])

Input fingerprint: f924633e9740d2446b4c5d327d6770ef2606c1135a4f105eab65444581f8764f
Missions: 3 Candidates: 12


mission,option,preference,sunlight_s,start_utc
crew,crew-1,12.0,0.0,2026-11-04T00:00:00+00:00
crew,crew-2,9.0,0.0,2026-11-04T01:00:00+00:00
crew,crew-3,6.0,0.0,2026-11-05T00:00:00+00:00
crew,crew-4,3.0,0.0,2026-11-05T01:00:00+00:00
robot-a,robot-a-1,8.0,0.0,2026-11-04T00:00:00+00:00
robot-a,robot-a-2,6.0,0.0,2026-11-04T01:00:00+00:00
robot-a,robot-a-3,4.0,0.0,2026-11-05T00:00:00+00:00
robot-a,robot-a-4,2.0,0.0,2026-11-05T01:00:00+00:00
robot-b,robot-b-1,4.0,0.0,2026-11-04T00:00:00+00:00
robot-b,robot-b-2,3.0,0.0,2026-11-04T01:00:00+00:00


## 3. Independent choices conflict
Each mission chooses its best option without joint scheduling. The displayed score of an infeasible proposal is not achievable.

In [4]:
independent = select_independent(problem)
print(independent.plan.selected_option_ids)
print(independent.validation.status)
table([dict(code=i.code, options=i.option_ids, required=i.required, capacity=i.capacity,
    message=i.message) for i in independent.validation.issues])
assert independent.validation.status == "infeasible"

('crew-1', 'robot-a-1', 'robot-b-1')
infeasible


code,options,required,capacity,message
capacity,"('crew-1', 'robot-a-1', 'robot-b-1')",3,1,Simultaneous demand exceeds available channel capacity.


## 4. Exact joint reference and antenna occupancy
The exact solver checks every assignment. With alternative 3 geometrically blocked, the three missions must occupy the three remaining distinct windows. No required mission is dropped. Half-open intervals allow an end and start to coincide without double booking.

In [5]:
exact = solve_exact(problem)
assert exact.status == "optimal" and exact.evaluated_combinations == 64
assert exact.feasible_count == 6
print("Best plan:", exact.plan.selected_option_ids, "Exact score:", exact.validation.exact_score)
timeline = campaign_timeline(problem, exact.plan)
table([dict(option=r["option_id"], start_utc=r["window"]["start_utc"],
    end_utc=r["window"]["end_utc"]) for r in timeline["rows"]])
table([dict(start_utc=r["window"]["start_utc"], end_utc=r["window"]["end_utc"],
    demand=r["required_channels"], capacity=r["capacity_channels"], conflict=r["conflict"])
    for r in timeline["occupancy"] if r["required_channels"]])

Best plan: ('crew-1', 'robot-a-2', 'robot-b-4') Exact score: 19/4


option,start_utc,end_utc
crew-1,2026-11-04T00:00:00Z,2026-11-04T00:30:00Z
robot-a-2,2026-11-04T01:00:00Z,2026-11-04T01:30:00Z
robot-b-4,2026-11-05T01:00:00Z,2026-11-05T01:30:00Z


start_utc,end_utc,demand,capacity,conflict
2026-11-04T00:00:00Z,2026-11-04T00:30:00Z,1.0,1.0,False
2026-11-04T01:00:00Z,2026-11-04T01:30:00Z,1.0,1.0,False
2026-11-05T01:00:00Z,2026-11-05T01:30:00Z,1.0,1.0,False


## 5. Move the crew decision and replan
Each choice rebuilds the source-bound scenario; clocks are not merely shifted on a plot. A fixed option is a hard constraint. Alternative 3 has no DSN geometry and therefore cannot be made feasible by a high score.

In [6]:
changes = []
for index in (0, 1, 2, 3, 4):
    revised = build_campaign(CampaignRequest(crew_option=index), evidence)
    solved = solve_exact(revised)
    if solved.plan and index:
        assert f"crew-{index}" in solved.plan.selected_option_ids
    changes.append(dict(crew_option=index or "automatic", status=solved.status,
        plan=solved.plan.selected_option_ids if solved.plan else None,
        score=solved.validation.score if solved.validation else None,
        fingerprint=fingerprint(revised)))
table(changes)
assert changes[3]["status"] == "infeasible"

crew_option,status,plan,score,fingerprint
automatic,optimal,"('crew-1', 'robot-a-2', 'robot-b-4')",4.75,f924633e9740d2446b4c5d327d6770ef2606c1135a4f105eab65444581f8764f
1,optimal,"('crew-1', 'robot-a-2', 'robot-b-4')",4.75,b4c6199939ab1210d1edf29b4aa55287e50325bd1d9147ece1ce8582ee3d4f5d
2,optimal,"('crew-2', 'robot-a-1', 'robot-b-4')",4.5,d7bd6a52733471d0b8f214e9bc1446a5dc8e82b4ef2b9f71a662e99f430c7b9f
3,infeasible,None,None,7a4c272f6d0fb773093263f4853c52d13e95400ea3d1a1e8140720beb3969e18
4,optimal,"('crew-4', 'robot-a-1', 'robot-b-2')",3.5,f979d48722f02203c430545d6900f6d50c0c2d5d3c89435d42a002fa5c8d3e69


## 6. Distinguish no solution from missing data
A closed antenna and a shared early deadline yield proven infeasibility at this bounded scale. Unknown station capacity yields incomplete input, not a zero-capacity assumption or a quantum success.

In [7]:
conditions = []
for mode in ("standard", "deadline", "closed", "unknown"):
    outcome = compare_campaign(CampaignRequest(mode=mode, evaluations=8, shots=128))
    conditions.append(dict(mode=mode, classical=outcome["comparison"]["classical"]["result"]["status"],
        quantum=outcome["comparison"]["quantum"]["status"], hardware_jobs=outcome["comparison"]["hardware_jobs"]))
table(conditions)
assert [r["classical"] for r in conditions] == ["optimal", "infeasible", "infeasible", "incomplete"]

mode,classical,quantum,hardware_jobs
standard,optimal,completed,0
deadline,infeasible,not_run,0
closed,infeasible,not_run,0
unknown,incomplete,not_run,0


## 7. Matched H8 comparison for this H9 input
The H8 engine is reused without changing the scientific input. Raw quantum samples retain invalid outcomes. Timings include mapping, exhaustive audits, classical tuning, sampling and validation; this is not a live race or a quantum speedup benchmark.

In [8]:
comparison = compare_campaign(settings)["comparison"]
assert comparison["input_fingerprint"] == digest
q = comparison["quantum"]["result"]
assert q["logical_qubits"] == 6
assert comparison["quantum"]["validation"]["status"] == "feasible"
table([dict(method=name, score=comparison[name]["score"], gap=comparison[name]["optimum_gap"],
    elapsed_s=comparison[name]["elapsed_s"]) for name in ("classical", "quantum")])
table([dict(basis=s["basis_index"], count=s["count"], feasible=s["feasible"], optimal=s["optimal"],
    options=s["selected_option_ids"]) for s in q["samples"] if s["count"]])

method,score,gap,elapsed_s
classical,4.75,0.0,0.007591207999212202
quantum,4.75,0.0,1.438574082996638


basis,count,feasible,optimal,options
0,1,False,False,"['crew-1', 'robot-a-1', 'robot-b-1']"
1,2,False,False,"['crew-2', 'robot-a-1', 'robot-b-1']"
3,5,False,False,"['crew-4', 'robot-a-1', 'robot-b-1']"
4,4,False,False,"['crew-1', 'robot-a-2', 'robot-b-1']"
5,1,False,False,"['crew-2', 'robot-a-2', 'robot-b-1']"
6,3,False,False,"['crew-3', 'robot-a-2', 'robot-b-1']"
7,15,True,False,"['crew-4', 'robot-a-2', 'robot-b-1']"
8,1,False,False,"['crew-1', 'robot-a-3', 'robot-b-1']"
9,5,False,False,"['crew-2', 'robot-a-3', 'robot-b-1']"
10,1,False,False,"['crew-3', 'robot-a-3', 'robot-b-1']"


## 8. Fixed local tuning sweep before hardware
Six predeclared trials: 1/2 layers, seeds 17/23/41, 80 evaluations and 256 samples each. Choose highest ideal feasible probability, then optimal probability, then lower depth and seed. All trials are reported. This selection is for this small instance only; it does not establish generalization or the lowest hardware error.

In [9]:
sweep = [run_qaoa(problem, depth=depth, seed=seed, evaluations=80, shots=256)
    for depth in (1, 2) for seed in (17, 23, 41)]
chosen = max(sweep, key=lambda r: (r.feasible_probability, r.optimal_probability, -r.depth, -r.seed))
table([dict(depth=r.depth, seed=r.seed, ideal_feasible=r.feasible_probability,
    ideal_optimal=r.optimal_probability, logical_CX=r.cx_gate_estimate,
    sampled_optimal=r.sampled_optimal_rate) for r in sweep])
print("Selected layers / seed:", chosen.depth, chosen.seed)
print("Uniform feasible / optimal:", chosen.uniform_feasible_probability, chosen.uniform_optimal_probability)

depth,seed,ideal_feasible,ideal_optimal,logical_CX,sampled_optimal
1,17,0.38651667625482056,0.06520952016082872,96,0.07421875
1,23,0.4121114108762095,0.0697197406953309,96,0.0859375
1,41,0.41472933693012615,0.07012871737453849,96,0.09765625
2,17,0.5778262133267403,0.09949178744349856,192,0.109375
2,23,0.4851455826316608,0.08282482970718946,192,0.1015625
2,41,0.678607184218657,0.11744047799654325,192,0.12890625


Selected layers / seed: 2 41
Uniform feasible / optimal: 0.09375 0.015625


## 9. Independent circuit check
Qiskit statevector probabilities are compared with the NumPy implementation before provider preparation. Provider routing and physical layout can add swaps and gates; six logical qubits do not mean a cheap or low-error device circuit.

In [10]:
from quantum.openquantum_bridge import build_qaoa_circuit
from qiskit.quantum_info import Statevector
circuit = build_qaoa_circuit(problem, chosen.gammas, chosen.betas, measure=False)
np.testing.assert_allclose(Statevector.from_instruction(circuit).probabilities(),
    [sample.probability for sample in chosen.samples], atol=1e-11, rtol=0)
print("Independent statevector check passed. Logical operations:", dict(circuit.count_ops()))

Independent statevector check passed. Logical operations: {'cx': 192, 'rz': 72, 'rx': 12, 'h': 6}


## 10. Recorded H9 hardware, when available
Loading the record performs no network request. Every observed bitstring is independently validated against this exact scenario. Hardware frequencies, the ideal distribution and classical optimum are separate evidence. A single job is not a performance advantage claim; training happened locally.

New submission is deliberately absent from Run All. The separate H7 gateway requires a fresh price, Spark-only coverage, verified Auto Refill off, and a durable receipt. H9's authorized total trial cap is one Spark. Do not change trial IDs to bypass the cap.

In [11]:
hardware = campaign_hardware()
print("Hardware record status:", hardware["status"], "New jobs submitted:", hardware["new_jobs_submitted"])
if hardware["status"] == "completed":
    assert hardware["input_fingerprint"] == digest
    assert sum(hardware["counts"].values()) == hardware["shots"]
    table([{k: hardware.get(k) for k in ("job_id", "shots", "credits_spent", "sampled_feasible_rate", "sampled_optimal_rate", "optimum_gap")}])
    table([dict(bitstring=k, count=v) for k, v in sorted(hardware["counts"].items())])
    if hardware["best_plan"]:
        assert validate_plan(Plan.model_validate(hardware["best_plan"]), problem).status == "feasible"
print("Notebook execution spent 0 credits and submitted 0 provider jobs.")

Hardware record status: completed New jobs submitted: 0


job_id,shots,credits_spent,sampled_feasible_rate,sampled_optimal_rate,optimum_gap
03f3af9d-ddfe-4540-a1e5-d429aa9916d9,256,1.0,0.06640625,0.01171875,0.0


bitstring,count
000000,6
000001,2
000010,6
000011,6
000100,6
000101,4
000110,4
000111,6
001000,2
001001,1


Notebook execution spent 0 credits and submitted 0 provider jobs.
